In [0]:
# Databricks notebook source

# ==========================================================
# Gold Data Quality Checks
#
# Notebook:
# 03_gold_dq_checks
#
# Purpose:
# Validate Gold layer before reporting/dashboarding.
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC

CATALOG = "crypto_pipeline"

DAILY_TABLE = f"{CATALOG}.gold.daily_ohlc"
ROLLING_TABLE = f"{CATALOG}.gold.rolling_metrics"
TOP_MOVERS_TABLE = f"{CATALOG}.gold.top_movers"
DQ_TABLE = f"{CATALOG}.meta.dq_results"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create DQ Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {DQ_TABLE}

(

layer STRING,

check_name STRING,

status STRING,

failed_rows INT,

check_timestamp TIMESTAMP

)

USING DELTA

""")

daily_df = spark.table(DAILY_TABLE)
rolling_df = spark.table(ROLLING_TABLE)
top_df = spark.table(TOP_MOVERS_TABLE)

dq_logs = []

# ==========================================================
# Helper Function
# ==========================================================

def log_result(layer, check_name, failed_rows):

    status = "PASS" if failed_rows == 0 else "FAIL"

    dq_logs.append(

        Row(

            layer=layer,

            check_name=check_name,

            status=status,

            failed_rows=int(failed_rows),

            check_timestamp=datetime.now(UTC)

        )

    )

In [0]:
duplicates = (

    daily_df

    .groupBy(

        "coin_sk",

        "observation_date"

    )

    .count()

    .filter("count > 1")

)

dup_count = duplicates.count()

log_result(

    "gold",

    "Duplicate Daily OHLC",

    dup_count

)

In [0]:
invalid_high_low = (

    daily_df

    .filter(

        F.col("high_price") < F.col("low_price")

    )

)

invalid_count = invalid_high_low.count()

log_result(

    "gold",

    "High Lower Than Low",

    invalid_count

)

In [0]:
null_open = (

    daily_df

    .filter(

        F.col("open_price").isNull()

    )

)

null_open_count = null_open.count()

log_result(

    "gold",

    "Null Open Price",

    null_open_count

)

In [0]:
null_close = (

    daily_df

    .filter(

        F.col("close_price").isNull()

    )

)

null_close_count = null_close.count()

log_result(

    "gold",

    "Null Close Price",

    null_close_count

)

In [0]:
null_rolling = (

    rolling_df

    .filter(

        F.col("rolling_avg_price_7").isNull()

    )

)

null_rolling_count = null_rolling.count()

log_result(

    "gold",

    "Null Rolling Average",

    null_rolling_count

)

In [0]:
invalid_rank = (

    top_df

    .filter(

        F.col("rank_position") < 1

    )

)

invalid_rank_count = invalid_rank.count()

log_result(

    "gold",

    "Invalid Rank",

    invalid_rank_count

)

In [0]:
invalid_movement = (

    top_df

    .filter(

        ~F.col("movement").isin(

            "Gainer",

            "Loser",

            "Neutral"

        )

    )

)

invalid_move_count = invalid_movement.count()

log_result(

    "gold",

    "Invalid Movement",

    invalid_move_count

)

In [0]:
spark.createDataFrame(

    dq_logs

).write.mode(

    "append"

).saveAsTable(

    DQ_TABLE

)

display(

    spark.table(DQ_TABLE)

    .orderBy(

        F.desc("check_timestamp")

    )

)

layer,check_name,status,failed_rows,check_timestamp
gold,Invalid Movement,PASS,0,2026-07-22T11:55:40.259Z
gold,Invalid Rank,PASS,0,2026-07-22T11:55:33.695Z
gold,Null Rolling Average,PASS,0,2026-07-22T11:55:27.875Z
gold,Null Close Price,PASS,0,2026-07-22T11:55:22.611Z
gold,Null Open Price,PASS,0,2026-07-22T11:55:16.762Z
gold,High Lower Than Low,PASS,0,2026-07-22T11:55:11.016Z
gold,Duplicate Daily OHLC,PASS,0,2026-07-22T11:55:03.960Z
silver,Duplicate Coin SK,PASS,0,2026-07-22T10:44:29.333Z
silver,Null Coin SK,PASS,0,2026-07-22T10:44:28.528Z
silver,Negative Price,PASS,0,2026-07-22T10:44:27.865Z


In [0]:
failed = sum(

    row.failed_rows

    for row in dq_logs

)

if failed > 0:

    raise Exception(

        f"Gold Data Quality Failed. Failed Rows = {failed}"

    )

print("Gold Data Quality Passed Successfully.")

Gold Data Quality Passed Successfully.
